In [0]:
# Imports

from pyspark.sql.functions import (
    col, round as spark_round,
    first, current_timestamp, date_format
)
from delta.tables import DeltaTable

print("Imports successful")

In [0]:
# Read Bronze population and dim_country
df_bronze_pop  = spark.table("workspace.default.bronze_world_bank_population")
df_dim_country = spark.table("workspace.default.dim_country")

print(f"Bronze population rows: {df_bronze_pop.count()}")
print(f"dim_country rows:       {df_dim_country.count()}")

# Preview the long format — two indicators stacked
display(
    df_bronze_pop
    .select("country_code", "year", "indicator_name", "value")
    .filter(col("country_code") == "KE")
    .orderBy("year", "indicator_name")
    .limit(10)
)

In [0]:
# Pivot long to wide format

from pyspark.sql.functions import first
from pyspark.sql.types import LongType

df_pivoted_pop = (
    df_bronze_pop
    .groupBy("country_code", "year")
    .pivot(
        "indicator_name",
        ["total_population", "urban_population_pct"]
    )
    .agg(first("value"))
)

# Cast total_population to integer — population is always whole
# Round urban_population_pct to 2 decimal places
df_pivoted_pop = (
    df_pivoted_pop
    .withColumn("total_population",    col("total_population").cast(LongType()))
    .withColumn("urban_population_pct", spark_round(col("urban_population_pct"), 2))
)

print(f"Pivoted rows: {df_pivoted_pop.count()}")
print(f"\nSample — Kenya population trend:")
display(
    df_pivoted_pop
    .filter(col("country_code") == "KE")
    .orderBy("year")
)

In [0]:
# Join against dim_country
df_silver_pop = (
    df_pivoted_pop
    .join(
        df_dim_country.select("country_code", "country_iso3", "country_name", "region"),
        on="country_code",
        how="inner"
    )
    .select(
        "country_code",
        "country_iso3",
        "country_name",
        "region",
        "year",
        "total_population",
        "urban_population_pct"
    )
)

df_silver_pop = df_silver_pop.withColumn(
    "updated_at",
    date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)

print(f"Silver population rows after join: {df_silver_pop.count()}")
print(f"\nSample:")
display(
    df_silver_pop
    .filter(col("country_code") == "KE")
    .orderBy("year")
)

In [0]:
# First-time write
TABLE_NAME = "workspace.default.silver_fact_population"

(df_silver_pop.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Silver population table created: {TABLE_NAME}")
print(f"Row count: {spark.table(TABLE_NAME).count()}")

In [0]:
# Cell 6 — MERGE INTO

TABLE_NAME = "workspace.default.silver_fact_population"

delta_target = DeltaTable.forName(spark, TABLE_NAME)

(
    delta_target.alias("target")
    .merge(
        df_silver_pop.alias("source"),
        "target.country_code = source.country_code AND target.year = source.year"
    )
    .whenMatchedUpdate(set={
        "total_population":    "source.total_population",
        "urban_population_pct":"source.urban_population_pct",
        "country_iso3":        "source.country_iso3",
        "country_name":        "source.country_name",
        "region":              "source.region",
        "updated_at":          "source.updated_at",
    })
    .whenNotMatchedInsert(values={
        "country_code":        "source.country_code",
        "country_iso3":        "source.country_iso3",
        "country_name":        "source.country_name",
        "region":              "source.region",
        "year":                "source.year",
        "total_population":    "source.total_population",
        "urban_population_pct":"source.urban_population_pct",
        "updated_at":          "source.updated_at",
    })
    .execute()
)

print("MERGE complete")
print(f"Row count after MERGE: {spark.table(TABLE_NAME).count()}")

In [0]:
# Cell 7 — Verify

from pyspark.sql.functions import max, min, avg

df_check = spark.table("workspace.default.silver_fact_population")

print(f"Total rows: {df_check.count()}")

print("\nLatest population by country (2023):")
display(
    df_check
    .filter(col("year") == 2023)
    .select("country_name", "region", "total_population", "urban_population_pct")
    .orderBy("total_population", ascending=False)
)

In [0]:
# Full Silver audit

silver_tables = [
    ("dim_country",            15),
    ("silver_fact_gdp",       210),
    ("silver_fact_climate",  2520),
    ("silver_fact_food",      210),
    ("silver_fact_population", 210),
]

print("Silver layer audit:")
print("-" * 50)
all_passed = True
for table, expected in silver_tables:
    actual = spark.table(f"workspace.default.{table}").count()
    status = "-" if actual >= expected - 10 else "!"
    if actual < expected - 10:
        all_passed = False
    print(f"{status} {table}: {actual} rows (expected ~{expected})")

print("-" * 50)
print(f"Silver layer: {'COMPLETE' if all_passed else 'ISSUES FOUND'}")